## Pose Estimation using ArUco markers

In [2]:
import cv2
import numpy as np

OpenCV version: 5.0.0


In [3]:
MARKER_SIZE_CM = 5.0


focal_length = 600
camera_matrix = np.array([[focal_length, 0, 320],
                          [0, focal_length, 240],
                          [0, 0, 1]], dtype=float)
dist_coeffs = np.zeros((5, 1))   # taking the lens distortion to be null


half = MARKER_SIZE_CM / 2
marker_points_3d = np.array([[-half,  half, 0],
                             [ half,  half, 0],
                             [ half, -half, 0],
                             [-half, -half, 0]], dtype=np.float32)

In [4]:
detector_settings = cv2.aruco.DetectorParameters()

dict_4x4 = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_1000)
dict_5x5 = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_1000)
dict_6x6 = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_1000)

detector_4x4 = cv2.aruco.ArucoDetector(dict_4x4, detector_settings)
detector_5x5 = cv2.aruco.ArucoDetector(dict_5x5, detector_settings)
detector_6x6 = cv2.aruco.ArucoDetector(dict_6x6, detector_settings)


all_detectors = [("4x4", detector_4x4), ("5x5", detector_5x5), ("6x6", detector_6x6)]
print("detector has been created ")

Detectors ready!


In [5]:
def process_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)   # detection on on a black & white picture
    tags_found = 0

    for tag_name, detector in all_detectors:
        corners, ids, rejected = detector.detectMarkers(gray)

        if ids is None:
            continue

        cv2.aruco.drawDetectedMarkers(frame, corners, ids)

        ids = ids.flatten()

        for i in range(len(ids)):
            marker_id = ids[i]
            marker_corners = corners[i].reshape(4, 2)

            ok, rvec, tvec = cv2.solvePnP(marker_points_3d, marker_corners,
                                          camera_matrix, dist_coeffs,
                                          flags=cv2.SOLVEPNP_IPPE_SQUARE)
            if not ok:
                continue

            tags_found = tags_found + 1


            cv2.drawFrameAxes(frame, camera_matrix, dist_coeffs, rvec, tvec, MARKER_SIZE_CM / 2)


            x = round(tvec[0][0], 1)
            y = round(tvec[1][0], 1)
            z = round(tvec[2][0], 1)


            rotation_matrix, _ = cv2.Rodrigues(rvec)
            angles = cv2.RQDecomp3x3(rotation_matrix)[0]
            tilt_x = angles[0]
            if tilt_x < 0:
                tilt_x = tilt_x + 180
            else:
                tilt_x = tilt_x - 180
            tilt_y = angles[1]
            tilt_z = angles[2]


            text_x = int(marker_corners[0][0])
            text_y = int(marker_corners[0][1]) - 45
            if text_y < 20:
                text_y = 20

            text1 = "ID " + str(marker_id) + " (" + tag_name + ")"
            text2 = "x=" + str(x) + " y=" + str(y) + " depth=" + str(z) + " cm"
            text3 = "tilt: " + str(round(tilt_x)) + ", " + str(round(tilt_y)) + ", " + str(round(tilt_z)) + " deg"

            cv2.putText(frame, text1, (text_x, text_y),      cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
            cv2.putText(frame, text2, (text_x, text_y + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
            cv2.putText(frame, text3, (text_x, text_y + 36), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

    return frame, tags_found

## google colab to webcam linking

In [7]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode

def start_camera():
    js = Javascript("""
    var video, div = null, stream, captureCanvas, imgElement, labelElement;
    var pendingResolve = null, shutdown = false;

    function removeDom() {
      stream.getVideoTracks()[0].stop();
      video.remove(); div.remove();
      video = null; div = null; stream = null;
      imgElement = null; captureCanvas = null; labelElement = null;
    }
    function onAnimationFrame() {
      if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8);
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }
    async function createDom() {
      if (div !== null) { return stream; }
      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status:</span>";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia({video: true});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML = '<span style="color: red; font-weight: bold;">Click here or on the video to stop</span>';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640;
      captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame);
      return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) { removeDom(); shutdown = false; return ''; }
      stream = await createDom();
      if (label != "") { labelElement.innerHTML = label; }
      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }
      var result = await new Promise(function(resolve, reject) { pendingResolve = resolve; });
      shutdown = false;
      return {'img': result};
    }
    """)
    display(js)

def get_frame(status_text, picture_to_show):
    return eval_js('stream_frame("{}", "{}")'.format(status_text, picture_to_show))

def browser_to_opencv(js_reply):
    jpeg_bytes = b64decode(js_reply.split(",")[1])
    return cv2.imdecode(np.frombuffer(jpeg_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)

def opencv_to_browser(picture):
    ok, buffer = cv2.imencode(".jpg", picture)
    return "data:image/jpeg;base64," + b64encode(buffer.tobytes()).decode("utf-8")

In [ ]:
start_camera()
status = "Starting camera..."
picture_to_show = ""

while True:
    reply = get_frame(status, picture_to_show)
    if not reply:
        break

    frame = browser_to_opencv(reply["img"])
    frame, count = process_frame(frame)
    status = str(count) + " tag(s) found"
    picture_to_show = opencv_to_browser(frame)

print("Program has enended")

<IPython.core.display.Javascript object>